In [55]:
# ============================================
# Load saved Statcast data
# ============================================

import pandas as pd
import numpy as np

data = pd.read_parquet("statcast_2021_2025.parquet")

print(data.shape)
data.head()

(3846144, 119)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,season
0,FF,2021-11-02,93.7,1.39,6.72,"Smith, Will",493329,519293,field_out,hit_into_play,...,1.39,0.57,-0.57,46.9,<NA>,<NA>,<NA>,<NA>,<NA>,2021
1,FF,2021-11-02,92.9,1.38,6.72,"Smith, Will",493329,519293,None,foul,...,1.32,0.9,-0.9,48.0,<NA>,<NA>,<NA>,<NA>,<NA>,2021
2,FF,2021-11-02,93.1,1.35,6.73,"Smith, Will",493329,519293,None,called_strike,...,1.14,0.81,-0.81,46.7,<NA>,<NA>,<NA>,<NA>,<NA>,2021
3,FF,2021-11-02,94.6,1.31,6.73,"Smith, Will",670541,519293,field_out,hit_into_play,...,1.32,0.85,0.85,47.7,<NA>,<NA>,<NA>,<NA>,<NA>,2021
4,FF,2021-11-02,93.6,1.31,6.8,"Smith, Will",670541,519293,None,ball,...,1.2,0.9,0.9,48.9,<NA>,<NA>,<NA>,<NA>,<NA>,2021


In [56]:
import pandas as pd

raw = pd.read_parquet("statcast_2021_2025.parquet")
print("num_cols:", len(raw.columns))
print(sorted(raw.columns))

num_cols: 119
['age_bat', 'age_bat_legacy', 'age_pit', 'age_pit_legacy', 'api_break_x_arm', 'api_break_x_batter_in', 'api_break_z_with_gravity', 'arm_angle', 'at_bat_number', 'attack_angle', 'attack_direction', 'away_score', 'away_team', 'ax', 'ay', 'az', 'babip_value', 'balls', 'bat_score', 'bat_score_diff', 'bat_speed', 'bat_win_exp', 'batter', 'batter_days_since_prev_game', 'batter_days_until_next_game', 'bb_type', 'break_angle_deprecated', 'break_length_deprecated', 'delta_home_win_exp', 'delta_pitcher_run_exp', 'delta_run_exp', 'des', 'description', 'effective_speed', 'estimated_ba_using_speedangle', 'estimated_slg_using_speedangle', 'estimated_woba_using_speedangle', 'events', 'fielder_2', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9', 'fld_score', 'game_date', 'game_pk', 'game_type', 'game_year', 'hc_x', 'hc_y', 'hit_distance_sc', 'hit_location', 'home_score', 'home_score_diff', 'home_team', 'home_win_exp', 'hyper_speed', 'if_fielding_

In [57]:
import pandas as pd
import numpy as np

raw = pd.read_parquet("statcast_2021_2025.parquet")

needed = [
    "batter","pitcher",
    "pitch_type","balls","strikes","outs_when_up","inning",
    "stand","p_throws",
    "on_1b","on_2b","on_3b",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "bat_score","fld_score",
    "events","des"   # <-- IMPORTANT: use 'des'
]

missing = [c for c in needed if c not in raw.columns]
print("missing:", missing)

df = raw[needed].copy()

# base state
df["runner_1b"] = df["on_1b"].notna().astype(np.int8)
df["runner_2b"] = df["on_2b"].notna().astype(np.int8)
df["runner_3b"] = df["on_3b"].notna().astype(np.int8)
df.drop(columns=["on_1b","on_2b","on_3b"], inplace=True)

# score -> run differential (batting - fielding)
df["run_diff"] = (df["bat_score"].fillna(0) - df["fld_score"].fillna(0)).astype(np.int16)
df.drop(columns=["bat_score","fld_score"], inplace=True)

print("df rebuilt:", df.shape)

missing: []
df rebuilt: (3846144, 20)


In [58]:
# ============================================
# Keep only modeling columns
# ============================================

cols = [
    "batter",
    "pitcher",
    "pitch_type",
    "balls",
    "strikes",
    "outs_when_up",
    "inning",
    "stand",
    "p_throws",
    "on_1b",
    "on_2b",
    "on_3b",
    "release_speed",
    "release_spin_rate",
    "pfx_x",
    "pfx_z",
    "zone",
    "events"
]

df = data[cols].copy()

print(df.shape)

(3846144, 18)


In [59]:
# ============================================
# Base state features
# ============================================

df["runner_1b"] = df["on_1b"].notna().astype(int)
df["runner_2b"] = df["on_2b"].notna().astype(int)
df["runner_3b"] = df["on_3b"].notna().astype(int)

df = df.drop(columns=["on_1b","on_2b","on_3b"])

In [64]:
import pandas as pd
import numpy as np

# =========================
# 1) Load raw parquet
# =========================
raw = pd.read_parquet("statcast_2021_2025.parquet")

# pitch ordering column
sort_col = "pitch_number" if "pitch_number" in raw.columns else ("pitch_num" if "pitch_num" in raw.columns else None)
if sort_col is None:
    raise ValueError("Could not find pitch order column: need 'pitch_number' or 'pitch_num'")

# =========================
# 2) Build PA-labeled pitch-state dataset
# =========================
needed = [
    "game_pk","at_bat_number",sort_col,
    "batter","pitcher",
    "pitch_type","balls","strikes","outs_when_up","inning",
    "stand","p_throws",
    "on_1b","on_2b","on_3b",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "bat_score","fld_score",
    "events","des"
]

missing = [c for c in needed if c not in raw.columns]
if missing:
    raise ValueError(f"Missing columns in parquet: {missing}")

tmp = raw[needed].copy()

# runners (current pitch state)
tmp["runner_1b"] = tmp["on_1b"].notna().astype(np.int8)
tmp["runner_2b"] = tmp["on_2b"].notna().astype(np.int8)
tmp["runner_3b"] = tmp["on_3b"].notna().astype(np.int8)
tmp.drop(columns=["on_1b","on_2b","on_3b"], inplace=True)

# score context (batting - fielding)
tmp["run_diff"] = (tmp["bat_score"].fillna(0) - tmp["fld_score"].fillna(0)).astype(np.int16)
tmp.drop(columns=["bat_score","fld_score"], inplace=True)

# sort to get final pitch of each PA
tmp = tmp.sort_values(["game_pk","at_bat_number",sort_col])

last_pitch = tmp.groupby(["game_pk","at_bat_number"], as_index=False).tail(1).copy()

def map_pa_outcome(events, des):
    ev = None if pd.isna(events) else str(events)
    d = "" if pd.isna(des) else str(des).lower()

    # hits
    if ev == "single": return "single"
    if ev == "double": return "double"
    if ev == "triple": return "triple"
    if ev == "home_run": return "hr"

    # walks / HBP
    if ev in ["walk","intent_walk","hit_by_pitch"]:
        return "walk"

    # strikeouts
    if "strikeout" in d or ev in ["strikeout","strikeout_double_play"]:
        return "strikeout"

    # other PA-ending events => out
    if ev is not None:
        return "out"

    return "none"

last_pitch["pa_outcome"] = [map_pa_outcome(e, d) for e, d in zip(last_pitch["events"], last_pitch["des"])]
last_pitch = last_pitch[last_pitch["pa_outcome"] != "none"][["game_pk","at_bat_number","pa_outcome"]].copy()

# merge PA outcome onto every pitch in the PA
tmp = tmp.merge(last_pitch, on=["game_pk","at_bat_number"], how="inner")

# drop raw terminal fields (we now have pa_outcome)
tmp.drop(columns=["events","des"], inplace=True)

print("PA-labeled pitch-state rows:", tmp.shape)
print("PA outcome distribution (%):")
print(tmp["pa_outcome"].value_counts(normalize=True).mul(100).round(2))

# =========================
# 3) Final modeling df
# =========================
features = [
    "pitcher","batter",
    "pitch_type",
    "balls","strikes",
    "outs_when_up","inning",
    "stand","p_throws",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "runner_1b","runner_2b","runner_3b",
    "run_diff",
]

# make sure key continuous features exist; drop rows missing essential physics
tmp["release_speed"] = pd.to_numeric(tmp["release_speed"], errors="coerce")
tmp["release_spin_rate"] = pd.to_numeric(tmp["release_spin_rate"], errors="coerce")
tmp["pfx_x"] = pd.to_numeric(tmp["pfx_x"], errors="coerce")
tmp["pfx_z"] = pd.to_numeric(tmp["pfx_z"], errors="coerce")
tmp["zone"] = pd.to_numeric(tmp["zone"], errors="coerce")

tmp = tmp.dropna(subset=["pitch_type","stand","p_throws","release_speed","pfx_x","pfx_z","zone"]).copy()

df = tmp[features + ["pa_outcome"]].copy()
print("Final modeling df:", df.shape)

PA-labeled pitch-state rows: (3844919, 22)
PA outcome distribution (%):
pa_outcome
out          39.40
strikeout    28.45
walk         13.39
single       12.06
double        3.70
hr            2.68
triple        0.32
Name: proportion, dtype: float64
Final modeling df: (3741396, 19)


In [65]:
# ============================================
# Encode categorical variables
# ============================================

from sklearn.preprocessing import LabelEncoder

encoders = {}

for col in ["pitch_type","stand","p_throws"]:

    le = LabelEncoder()

    df[col] = le.fit_transform(df[col].astype(str))

    encoders[col] = le

In [67]:
# ============================================
# FEATURES + TARGET (PA outcome)
# ============================================

features = [
    "pitcher",
    "batter",
    "pitch_type",
    "balls",
    "strikes",
    "outs_when_up",
    "inning",
    "stand",
    "p_throws",
    "release_speed",
    "release_spin_rate",
    "pfx_x",
    "pfx_z",
    "zone",
    "runner_1b",
    "runner_2b",
    "runner_3b",
    "run_diff",          # <-- add this back if you have it in df
]

X = df[features].copy()
y = df["pa_outcome"].copy()   # <-- THIS is the fix

# GPU CatBoost requires categorical cols to be int/str (not float)
cat_cols = ["pitcher","batter","pitch_type","stand","p_throws","zone"]
for c in cat_cols:
    X[c] = X[c].astype("string").fillna("NA")

In [ ]:
%pip install catboost

Note: you may need to restart the kernel to use updated packages.


In [69]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from catboost import CatBoostClassifier

# Features and target
X = df[features].copy()
y = df["pa_outcome"].copy()  # <-- FIXED

# Categorical columns (treat as strings for GPU safety)
cat_cols = ["pitcher","batter","pitch_type","stand","p_throws","zone"]
for c in cat_cols:
    X[c] = X[c].astype("string").fillna("NA")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=1200,
    learning_rate=0.05,
    depth=8,
    eval_metric="MultiClass",
    verbose=100,
    task_type="GPU",
    devices="0"
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

probs = model.predict_proba(X_test)

print("LogLoss:", log_loss(y_test, probs, labels=model.classes_))
print("Classes:", list(model.classes_))

0:	learn: 1.8900844	test: 1.8901198	best: 1.8901198 (0)	total: 138ms	remaining: 2m 44s
100:	learn: 1.3859076	test: 1.3855144	best: 1.3855144 (100)	total: 9.22s	remaining: 1m 40s
200:	learn: 1.3791026	test: 1.3793658	best: 1.3793658 (200)	total: 18.3s	remaining: 1m 31s
300:	learn: 1.3753550	test: 1.3765873	best: 1.3765873 (300)	total: 27.1s	remaining: 1m 21s
400:	learn: 1.3726570	test: 1.3748485	best: 1.3748485 (400)	total: 36s	remaining: 1m 11s
500:	learn: 1.3700117	test: 1.3732398	best: 1.3732398 (500)	total: 45.1s	remaining: 1m 2s
600:	learn: 1.3678503	test: 1.3719881	best: 1.3719881 (600)	total: 53.9s	remaining: 53.7s
700:	learn: 1.3656896	test: 1.3707970	best: 1.3707970 (700)	total: 1m 2s	remaining: 44.8s
800:	learn: 1.3634222	test: 1.3695629	best: 1.3695629 (800)	total: 1m 12s	remaining: 36.1s
900:	learn: 1.3613594	test: 1.3684851	best: 1.3684851 (900)	total: 1m 22s	remaining: 27.4s
1000:	learn: 1.3592290	test: 1.3673557	best: 1.3673557 (1000)	total: 1m 32s	remaining: 18.4s
1100:	

In [ ]:
from sklearn.metrics import accuracy_score

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test,preds))

Accuracy: 0.6249315777426129


In [ ]:
# ============================================
# Prediction function
# ============================================

def predict_at_bat(features_row):

    probs = model.predict_proba([features_row])[0]

    outcomes = model.classes_

    return dict(zip(outcomes,probs))

In [ ]:
import re
import numpy as np
import pandas as pd

from pybaseball import playerid_lookup

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss


# ============================================================
# LOAD
# ============================================================

DATA_PATH = "statcast_2021_2025.parquet"
raw = pd.read_parquet(DATA_PATH)
print("loaded:", raw.shape)


# ============================================================
# HELPERS: PLAYER NAME -> MLBAM ID
# ============================================================

def _split_name(name):
    name = name.strip()
    parts = name.split()
    if len(parts) < 2:
        raise ValueError("name must look like 'First Last'")
    first = parts[0]
    last = " ".join(parts[1:])
    return first, last

def _best_player_match(df):
    # prefer most recent mlb debut, else just first row
    # playerid_lookup returns columns like: key_mlbam, mlb_played_first, mlb_played_last
    if "mlb_played_last" in df.columns:
        df2 = df.copy()
        df2["mlb_played_last"] = pd.to_numeric(df2["mlb_played_last"], errors="coerce")
        df2 = df2.sort_values("mlb_played_last", ascending=False)
        return df2.iloc[0]
    return df.iloc[0]

def mlbam_id_from_name(name):
    first, last = _split_name(name)
    res = playerid_lookup(last, first)  # (last, first)
    if res is None or len(res) == 0:
        raise ValueError(f"could not find player: {name}")
    best = _best_player_match(res)
    pid = best.get("key_mlbam", None)
    if pd.isna(pid):
        raise ValueError(f"no MLBAM id found for: {name}")
    return int(pid)


# ============================================================
# FEATURE BUILD
# ============================================================

# columns we want from the raw statcast pull
# NOTE: score columns exist in statcast pulls for many seasons
needed = [
    "batter","pitcher",
    "pitch_type","balls","strikes","outs_when_up","inning",
    "stand","p_throws",
    "on_1b","on_2b","on_3b",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "bat_score","fld_score",
    "events"
]

missing = [c for c in needed if c not in raw.columns]
if missing:
    raise ValueError(
        "Your parquet is missing columns needed for this version:\n"
        + "\n".join(missing)
        + "\n\nIf you trimmed columns earlier, re-save the parquet with full statcast columns."
    )

df = raw[needed].copy()

# base state -> 0/1
df["runner_1b"] = df["on_1b"].notna().astype(np.int8)
df["runner_2b"] = df["on_2b"].notna().astype(np.int8)
df["runner_3b"] = df["on_3b"].notna().astype(np.int8)
df.drop(columns=["on_1b","on_2b","on_3b"], inplace=True)

# score -> run differential (batting - fielding)
df["run_diff"] = (df["bat_score"].fillna(0) - df["fld_score"].fillna(0)).astype(np.int16)
df.drop(columns=["bat_score","fld_score"], inplace=True)

# outcome mapping (plate appearance result buckets)
def map_events(x):
    if pd.isna(x):
        return "none"
    if x == "single":
        return "single"
    if x == "double":
        return "double"
    if x == "triple":
        return "triple"
    if x == "home_run":
        return "hr"
    if x in ["walk","intent_walk"]:
        return "walk"
    if x in ["strikeout","strikeout_double_play"]:
        return "strikeout"
    # everything else that ended a PA becomes "out"
    return "out"

df["outcome"] = df["events"].apply(map_events)
df = df[df["outcome"] != "none"].copy()
df.drop(columns=["events"], inplace=True)

# clean a bit
df["release_speed"] = pd.to_numeric(df["release_speed"], errors="coerce")
df["release_spin_rate"] = pd.to_numeric(df["release_spin_rate"], errors="coerce")
df["pfx_x"] = pd.to_numeric(df["pfx_x"], errors="coerce")
df["pfx_z"] = pd.to_numeric(df["pfx_z"], errors="coerce")
df["zone"] = pd.to_numeric(df["zone"], errors="coerce")

# drop rows missing core pitch physics
df = df.dropna(subset=["pitch_type","stand","p_throws","release_speed","pfx_x","pfx_z","zone"])
print("modeling df:", df.shape)


# ============================================================
# ENCODE CATEGORICALS
# ============================================================

enc = {}
for col in ["pitch_type","stand","p_throws"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    enc[col] = le


# ============================================================
# TRAIN MODEL
# ============================================================

features = [
    "pitcher","batter",
    "pitch_type",
    "balls","strikes",
    "outs_when_up","inning",
    "stand","p_throws",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "runner_1b","runner_2b","runner_3b",
    "run_diff",
]

X = df[features]
y = df["outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# HistGradientBoosting is fast + strong for large tabular data
model = HistGradientBoostingClassifier(
    max_depth=10,
    learning_rate=0.08,
    max_iter=200
)
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)
print("logloss:", log_loss(y_test, probs, labels=model.classes_))
print("classes:", list(model.classes_))


# ============================================================
# PARSING USER INPUTS
# ============================================================

def parse_count(count_str):
    # "3-1" -> (3,1)
    m = re.match(r"^\s*([0-3])\s*-\s*([0-2])\s*$", str(count_str))
    if not m:
        raise ValueError("count must look like '3-1'")
    return int(m.group(1)), int(m.group(2))

def parse_runners(runners_str):
    # "1;2" means runners on 1st and 2nd
    # allow: "", None, "0", "none"
    if runners_str is None:
        return 0, 0, 0
    s = str(runners_str).strip().lower()
    if s in ["", "0", "none", "empty"]:
        return 0, 0, 0
    parts = re.split(r"[;, ]+", s)
    r1 = 1 if "1" in parts else 0
    r2 = 1 if "2" in parts else 0
    r3 = 1 if "3" in parts else 0
    return r1, r2, r3

def parse_score(score_str):
    # "5:4" -> run_diff = 5 - 4 = +1 (assume batting team has 5)
    if score_str is None:
        return 0
    m = re.match(r"^\s*(-?\d+)\s*:\s*(-?\d+)\s*$", str(score_str))
    if not m:
        raise ValueError("score must look like '5:4'")
    bat = int(m.group(1))
    fld = int(m.group(2))
    return bat - fld

def parse_pitcher_blob(pitcher_blob):
    """
    Accept:
      "Tarik Skubal"
      "Tarik Skubal, 3-1"
      "Tarik Skubal, 3-1, 1;2"
      "Tarik Skubal, 3-1, 1;2, 5:4"
    Returns: name, count(optional), runners(optional), score(optional)
    """
    parts = [p.strip() for p in str(pitcher_blob).split(",")]
    name = parts[0]
    count = parts[1] if len(parts) >= 2 and parts[1] else None
    runners = parts[2] if len(parts) >= 3 and parts[2] else None
    score = parts[3] if len(parts) >= 4 and parts[3] else None
    return name, count, runners, score


# ============================================================
# CORE: BUILD A "SITUATIONAL" SAMPLE FOR THE MATCHUP
# ============================================================

def _mode(series):
    return series.value_counts(dropna=True).index[0]

def _build_situation_samples(
    batter_id,
    pitcher_id,
    balls,
    strikes,
    runner_1b,
    runner_2b,
    runner_3b,
    run_diff,
    n=1000
):
    """
    We simulate n "pitches seen in this situation" by sampling from the pitcher's
    real historical pitches in the same count (and backing off if too few).
    Then we overwrite batter/pitcher + situation fields, and average probabilities.
    """
    # start strict: pitcher + count
    pool = df[(df["pitcher"] == pitcher_id) & (df["balls"] == balls) & (df["strikes"] == strikes)]

    # backoff if pool is small (common with specific counts)
    if len(pool) < 500:
        pool = df[(df["pitcher"] == pitcher_id)]
    if len(pool) < 500:
        raise ValueError("not enough historical pitches for that pitcher in the dataset")

    # sample pitch shape + pitch_type + zone etc from pitcher history
    samp = pool.sample(n=min(n, len(pool)), replace=(len(pool) < n), random_state=42).copy()

    # lock in who is hitting/pitching
    samp["batter"] = batter_id
    samp["pitcher"] = pitcher_id

    # set situation
    samp["balls"] = balls
    samp["strikes"] = strikes
    samp["runner_1b"] = runner_1b
    samp["runner_2b"] = runner_2b
    samp["runner_3b"] = runner_3b
    samp["run_diff"] = run_diff

    # for stand/p_throws, use typical handedness for each player from history
    # batter stand = mode of batter's stand in dataset
    b_hist = df[df["batter"] == batter_id]
    p_hist = df[df["pitcher"] == pitcher_id]
    if len(b_hist) > 0:
        samp["stand"] = _mode(b_hist["stand"])
    if len(p_hist) > 0:
        samp["p_throws"] = _mode(p_hist["p_throws"])

    # clean up missing values before prediction
    samp = samp.fillna(0)

    return samp[features]


def predict_matchup(
    batter_name,
    pitcher_blob,
    score=None,
    count=None,
    runners=None,
    n_sims=1000
):
    """
    Usage styles:

    predict_matchup("Aaron Judge", "Tarik Skubal, 3-1, 1;2", "5:4")
    predict_matchup("Aaron Judge", "Tarik Skubal", count="3-1", runners="1;2", score="5:4")

    Notes:
    - If pitcher_blob already contains count/runners/score, those override kwargs.
    - score uses "batting:fielding" assumption.
    """

    p_name, p_count, p_runners, p_score = parse_pitcher_blob(pitcher_blob)

    # pitcher_blob overrides explicit kwargs if present
    if p_count is not None:
        count = p_count
    if p_runners is not None:
        runners = p_runners
    if p_score is not None:
        score = p_score

    if count is None:
        count = "0-0"
    balls, strikes = parse_count(count)

    r1, r2, r3 = parse_runners(runners)
    rd = parse_score(score)

    batter_id = mlbam_id_from_name(batter_name)
    pitcher_id = mlbam_id_from_name(p_name)

    Xsim = _build_situation_samples(
        batter_id=batter_id,
        pitcher_id=pitcher_id,
        balls=balls,
        strikes=strikes,
        runner_1b=r1,
        runner_2b=r2,
        runner_3b=r3,
        run_diff=rd,
        n=n_sims
    )

    prob_mat = model.predict_proba(Xsim)
    avg_probs = prob_mat.mean(axis=0)

    out = pd.Series(avg_probs, index=model.classes_).sort_values(ascending=False)

    # nice display
    result = (out * 100).round(2)

    return {
        "batter": batter_name,
        "pitcher": p_name,
        "count": f"{balls}-{strikes}",
        "runners": {"1b": r1, "2b": r2, "3b": r3},
        "run_diff": rd,
        "n_sims": len(Xsim),
        "probs_percent": result.to_dict()
    }


# ============================================================
# EXAMPLES
# ============================================================

# 1) all-in-one pitcher blob + separate score
# print(predict_matchup("Aaron Judge", "Tarik Skubal, 3-1, 1;2", "5:4", n_sims=2000))

# 2) kwargs style
# print(predict_matchup("Aaron Judge", "Tarik Skubal", count="3-1", runners="1;2", score="5:4", n_sims=2000))

loaded: (3846144, 119)
modeling df: (959666, 19)
logloss: 1.0033447131011703
classes: ['double', 'hr', 'out', 'single', 'strikeout', 'triple', 'walk']


In [ ]:
predict_matchup("Aaron Judge","Tarik Skubal, 3-1, 1;2","5:4")

{'batter': 'Aaron Judge',
 'pitcher': 'Tarik Skubal',
 'count': '3-1',
 'runners': {'1b': 1, '2b': 1, '3b': 0},
 'run_diff': 1,
 'n_sims': 1000,
 'probs_percent': {'out': 41.1,
  'walk': 33.0,
  'single': 12.41,
  'hr': 7.83,
  'double': 5.3,
  'triple': 0.34,
  'strikeout': 0.02}}

In [ ]:
import numpy as np
import pandas as pd

def simulate_matchup(
    batter_name,
    pitcher_name,
    count="0-0",
    runners=None,
    score=None,
    sims=10000,
    random_state=42
):

    # get probability distribution from model
    res = predict_matchup(
        batter_name,
        f"{pitcher_name}, {count}" + (f", {runners}" if runners else ""),
        score,
        n_sims=min(2000, sims)
    )

    probs_pct = res["probs_percent"]

    outcomes = list(probs_pct.keys())
    probs = np.array(list(probs_pct.values())) / 100.0

    rng = np.random.default_rng(random_state)

    draws = rng.choice(outcomes, size=sims, p=probs)

    sim_counts = pd.Series(draws).value_counts().reindex(outcomes, fill_value=0)

    sim_pct = (sim_counts / sims * 100).round(2)

    return {
        "batter": batter_name,
        "pitcher": pitcher_name,
        "count": count,
        "runners": runners,
        "score": score,
        "simulated_results_percent": sim_pct.to_dict(),
        "model_probs_percent": probs_pct
    }

In [ ]:
simulate_matchup("Aaron Judge","Tarik Skubal", sims=10000)

{'batter': 'Aaron Judge',
 'pitcher': 'Tarik Skubal',
 'count': '0-0',
 'runners': None,
 'score': None,
 'simulated_results_percent': {'out': 67.71,
  'single': 19.79,
  'double': 6.23,
  'hr': 5.88,
  'triple': 0.35,
  'strikeout': 0.03,
  'walk': 0.01},
 'model_probs_percent': {'out': 67.04,
  'single': 20.24,
  'double': 6.34,
  'hr': 6.06,
  'triple': 0.29,
  'strikeout': 0.02,
  'walk': 0.01}}

In [ ]:
import pandas as pd
import numpy as np

raw = pd.read_parquet("statcast_2021_2025.parquet")

# ---- required columns for grouping a plate appearance ----
id_cols = ["game_pk", "at_bat_number"]
sort_col_candidates = ["pitch_number", "pitch_num"]  # sometimes different names
sort_col = next((c for c in sort_col_candidates if c in raw.columns), None)

missing_ids = [c for c in id_cols if c not in raw.columns]
if missing_ids:
    raise ValueError(f"Missing PA id columns: {missing_ids}. Columns available: {list(raw.columns)[:30]} ...")

if sort_col is None:
    raise ValueError("Could not find pitch order column (pitch_number/pitch_num). Check your parquet columns.")

# ---- we will use 'des' for strikeouts, and 'events' for hits/walks ----
need = [
    "game_pk","at_bat_number",sort_col,
    "batter","pitcher",
    "pitch_type","balls","strikes","outs_when_up","inning",
    "stand","p_throws",
    "on_1b","on_2b","on_3b",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "bat_score","fld_score",
    "events","des"
]

missing = [c for c in need if c not in raw.columns]
if missing:
    raise ValueError(f"Missing needed columns: {missing}")

df = raw[need].copy()

# ---- base state features (state at CURRENT pitch) ----
df["runner_1b"] = df["on_1b"].notna().astype(np.int8)
df["runner_2b"] = df["on_2b"].notna().astype(np.int8)
df["runner_3b"] = df["on_3b"].notna().astype(np.int8)
df.drop(columns=["on_1b","on_2b","on_3b"], inplace=True)

# ---- score context ----
df["run_diff"] = (df["bat_score"].fillna(0) - df["fld_score"].fillna(0)).astype(np.int16)
df.drop(columns=["bat_score","fld_score"], inplace=True)

# ---- get FINAL pitch of each PA ----
df_sorted = df.sort_values(["game_pk","at_bat_number",sort_col]).copy()
last_pitch = df_sorted.groupby(["game_pk","at_bat_number"], as_index=False).tail(1).copy()

# ---- map final PA outcome using (events + des) from LAST pitch ----
def map_pa_outcome(events, des):
    ev = None if pd.isna(events) else str(events)
    d = "" if pd.isna(des) else str(des).lower()

    if ev == "single":
        return "single"
    if ev == "double":
        return "double"
    if ev == "triple":
        return "triple"
    if ev == "home_run":
        return "hr"

    if ev in ["walk", "intent_walk", "hit_by_pitch"]:
        return "walk"

    # strikeouts: 'des' catches most
    if "strikeout" in d or ev in ["strikeout", "strikeout_double_play"]:
        return "strikeout"

    # if the PA ended with some other event, count it as out
    if ev is not None:
        return "out"

    return "none"

last_pitch["pa_outcome"] = [
    map_pa_outcome(e, d) for e, d in zip(last_pitch["events"], last_pitch["des"])
]

# keep only PAs with a valid terminal outcome
last_pitch = last_pitch[last_pitch["pa_outcome"] != "none"][["game_pk","at_bat_number","pa_outcome"]].copy()

# ---- attach PA outcome to EVERY pitch in that PA ----
df = df.merge(last_pitch, on=["game_pk","at_bat_number"], how="inner")

# drop raw outcome columns (we only need pa_outcome now)
df.drop(columns=["events","des"], inplace=True)

print("Pitch-state rows with PA labels:", df.shape)
print(df["pa_outcome"].value_counts(normalize=True).mul(100).round(2))

Pitch-state rows with PA labels: (3844919, 22)
pa_outcome
out          39.40
strikeout    28.45
walk         13.39
single       12.06
double        3.70
hr            2.68
triple        0.32
Name: proportion, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from catboost import CatBoostClassifier

# Features (same idea as before, but target is now pa_outcome)
features = [
    "pitcher","batter",
    "pitch_type",
    "balls","strikes",
    "outs_when_up","inning",
    "stand","p_throws",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "runner_1b","runner_2b","runner_3b",
    "run_diff",
]

X = df[features].copy()
y = df["pa_outcome"].copy()

# GPU CatBoost: categorical cols must be str/int (NOT float)
cat_cols = ["pitcher","batter","pitch_type","stand","p_throws","zone"]
for c in cat_cols:
    X[c] = X[c].astype("string").fillna("NA")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=1200,
    learning_rate=0.05,
    depth=8,
    eval_metric="MultiClass",
    verbose=100,
    task_type="GPU",
    devices="0"
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

probs = model.predict_proba(X_test)
print("LogLoss:", log_loss(y_test, probs, labels=model.classes_))
print("Classes:", list(model.classes_))

KeyError: 'pa_outcome'

In [ ]:
predict_matchup("Aaron Judge","Tarik Skubal, 3-1, 1;2","5:4")

{'batter': 'Aaron Judge',
 'pitcher': 'Tarik Skubal',
 'count': '3-1',
 'runners': {'1b': 1, '2b': 1, '3b': 0},
 'run_diff': 1,
 'n_sims': 1000,
 'probs_percent': {'out': 39.8,
  'walk': 34.18,
  'single': 12.68,
  'hr': 7.57,
  'double': 5.46,
  'triple': 0.29,
  'strikeout': 0.02}}